In [ ]:
def read_iob2(filepath, masked_test=False):
    sentences = []
    labels = []

    current_tokens = []
    current_labels = []

    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_labels)
                    current_tokens = []
                    current_labels = []
                continue

            if line.startswith("#"):
                continue

            parts = line.split()

            token = parts[1]

            if masked_test:
                tag = "O"
            else:
                tag = parts[2]

            current_tokens.append(token)
            current_labels.append(tag)

    if current_tokens:
        sentences.append(current_tokens)
        labels.append(current_labels)

    return {"tokens": sentences, "ner_tags": labels}

In [7]:
train_data = read_iob2("en_ewt-ud-train.iob2")
dev_data = read_iob2("en_ewt-ud-dev.iob2")

print("Train sentences:", len(train_data["tokens"]))
print("Dev sentences:", len(dev_data["tokens"]))
print()
print("First train sentence:")
print(train_data["tokens"][0])
print(train_data["ner_tags"][0])

Train sentences: 12543
Dev sentences: 2001

First train sentence:
['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?']
['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O']


In [9]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_dict(train_data),
    "validation": Dataset.from_dict(dev_data),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 12543
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 2001
    })
})

In [10]:
all_labels = set()

for split in ["train", "validation"]:
    for sentence_labels in dataset[split]["ner_tags"]:
        all_labels.update(sentence_labels)

label_list = sorted(all_labels)

print(label_list)
print("Number of labels:", len(label_list))

['B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER', 'O']
Number of labels: 7


In [11]:
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'B-LOC': 0, 'B-ORG': 1, 'B-PER': 2, 'I-LOC': 3, 'I-ORG': 4, 'I-PER': 5, 'O': 6}


In [12]:
def encode_labels(example):
    example["labels"] = [label2id[label] for label in example["ner_tags"]]
    return example

In [13]:
dataset = dataset.map(encode_labels)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [14]:
print(dataset["train"][0])

{'tokens': ['Where', 'in', 'the', 'world', 'is', 'Iguazu', '?'], 'ner_tags': ['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O'], 'labels': [6, 6, 6, 6, 6, 0, 6]}


In [16]:
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

In [17]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LG\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [18]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    aligned_labels = []

    for i, labels in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        aligned_labels.append(label_ids)

    tokenized_inputs["labels"] = aligned_labels
    return tokenized_inputs

In [19]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/12543 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_datasets["train"][0].keys())
print(tokenized_datasets["train"][0]["input_ids"][:20])
print(tokenized_datasets["train"][0]["labels"][:20])
#Here I finished preprocessing part and move to ML Training

dict_keys(['tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])
[101, 2777, 1107, 1103, 1362, 1110, 146, 13855, 10337, 136, 102]
[-100, 6, 6, 6, 6, 6, 0, -100, -100, 6, -100]


In [21]:
model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized beca

In [22]:
data_collator = DataCollatorForTokenClassification(tokenizer)

In [23]:
from seqeval.metrics import precision_score, recall_score, f1_score, accuracy_score
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):
        preds = []
        labs = []

        for pred, lab in zip(prediction, label):
            if lab != -100:
                preds.append(label_list[pred])
                labs.append(label_list[lab])

        true_predictions.append(preds)
        true_labels.append(labs)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
        "accuracy": accuracy_score(true_labels, true_predictions),
    }

In [24]:
training_args = TrainingArguments(
    output_dir="./ewt_baseline",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1"
)

In [28]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [29]:
trainer.train()

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.030993,0.056774,0.768780,0.815735,0.791562,0.983061
2,0.019862,0.059507,0.786080,0.795031,0.790530,0.983816
3,0.009014,0.063473,0.797970,0.813665,0.805741,0.984532


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=2352, training_loss=0.03236253582001315, metrics={'train_runtime': 7288.0945, 'train_samples_per_second': 5.163, 'train_steps_per_second': 0.323, 'total_flos': 1037340407838834.0, 'train_loss': 0.03236253582001315, 'epoch': 3.0})

In [1]:
trainer.save_model("./ewt_baseline_model")
tokenizer.save_pretrained("./ewt_baseline_model")

NameError: name 'trainer' is not defined

In [30]:
predictions, labels, metrics = trainer.predict(tokenized_datasets["validation"])

import numpy as np
pred_ids = np.argmax(predictions, axis=2)

pred_label_sequences = []

for pred_seq, gold_seq in zip(pred_ids, labels):
    cur_preds = []
    for p, g in zip(pred_seq, gold_seq):
        if g != -100:
            cur_preds.append(label_list[p])
    pred_label_sequences.append(cur_preds)

print(pred_label_sequences[0])

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


['O', 'O', 'O', 'O', 'O', 'O', 'B-LOC', 'I-LOC', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [31]:
def write_pred_file_from_original(original_path, output_path, pred_sequences):
    sent_idx = 0
    tok_idx = 0

    with open(original_path, "r", encoding="utf-8") as fin, open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            stripped = line.strip()

            if stripped == "":
                fout.write("\n")
                if sent_idx < len(pred_sequences):
                    sent_idx += 1
                    tok_idx = 0
                continue

            if stripped.startswith("#"):
                fout.write(line)
                continue

            parts = stripped.split("\t")
            parts[2] = pred_sequences[sent_idx][tok_idx]
            fout.write("\t".join(parts) + "\n")
            tok_idx += 1

In [32]:
write_pred_file_from_original(
    "en_ewt-ud-dev.iob2",
    "dev_predictions.iob2",
    pred_label_sequences
)

In [33]:
!python span_f1.py en_ewt-ud-dev.iob2 dev_predictions.iob2

recall:    0.8178053830227743
precision: 0.8263598326359832
slot-f1:   0.8220603537981268

unlabeled
ul_recall:    0.8571428571428571
ul_precision: 0.8661087866108786
ul_slot-f1:   0.8616024973985431

loose (partial overlap with same label)
l_recall:    0.8623188405797102
l_precision: 0.8661087866108786
l_slot-f1:   0.8642096584629958


In [34]:
test_data = read_iob2("en_ewt-ud-test-masked.iob2")
print("Test sentences:", len(test_data["tokens"]))
print(test_data["tokens"][0])
print(test_data["ner_tags"][0])

Test sentences: 2077
['What', 'is', 'this', 'Miramar', '?']
['O', 'O', 'O', 'O', 'O']


In [35]:
from datasets import Dataset

test_dataset = Dataset.from_dict(test_data)
test_dataset

Dataset({
    features: ['tokens', 'ner_tags'],
    num_rows: 2077
})

In [36]:
def encode_labels(example):
    example["labels"] = [label2id[label] for label in example["ner_tags"]]
    return example

test_dataset = test_dataset.map(encode_labels)

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [37]:
tokenized_test_dataset = test_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_test_dataset[0].keys())
print(tokenized_test_dataset[0]["labels"][:20]) 

dict_keys(['tokens', 'ner_tags', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'])
[-100, 6, 6, 6, 6, -100, 6, -100]


In [39]:
test_predictions = trainer.predict(tokenized_test_dataset)

c:\Users\LG\anaconda3\envs\ner\lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [40]:
import numpy as np

test_pred_ids = np.argmax(test_predictions.predictions, axis=2)

test_pred_label_sequences = []

for pred_seq, gold_seq in zip(test_pred_ids, test_predictions.label_ids):
    cur_preds = []
    for p, g in zip(pred_seq, gold_seq):
        if g != -100:
            cur_preds.append(label_list[p])
    test_pred_label_sequences.append(cur_preds)

print(test_pred_label_sequences[0])

['O', 'O', 'O', 'O', 'O']


In [41]:
write_pred_file_from_original(
    "en_ewt-ud-test-masked.iob2",
    "test_predictions.iob2",
    test_pred_label_sequences
)

In [42]:
with open("test_predictions.iob2", "r", encoding="utf-8") as f:
    for i in range(20):
        print(repr(f.readline()))

'# newdoc id = answers-20080426140040AA4YiX5_ans\n'
'# sent_id = answers-20080426140040AA4YiX5_ans-0001\n'
'# text = What is this Miramar?\n'
'1\tWhat\tO\t-\t-\n'
'2\tis\tO\t-\t-\n'
'3\tthis\tO\t-\t-\n'
'4\tMiramar\tO\t-\tstephen\n'
'5\t?\tO\t-\t-\n'
'\n'
'# sent_id = answers-20080426140040AA4YiX5_ans-0002\n'
'# text = It is a place in Argentina lol\n'
'1\tIt\tO\t-\t-\n'
'2\tis\tO\t-\t-\n'
'3\ta\tO\t-\t-\n'
'4\tplace\tO\t-\t-\n'
'5\tin\tO\t-\t-\n'
'6\tArgentina\tB-LOC\t-\tstephen\n'
'7\tlol\tO\t-\t-\n'
'\n'
'# newdoc id = answers-20081218053636AA9vV0u_ans\n'


In [44]:
print(test_pred_label_sequences[1])

['O', 'O', 'O', 'O', 'O', 'B-LOC', 'O']
